# Modulo 5 · Lezione 7 — Progetto: API di gestione formazioni

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa costruiamo oggi?

Un'API REST completa per gestire corsi, iscrizioni e partecipanti di un centro di formazione.

**Stack completo del modulo applicato:**

```
FastAPI         → endpoint REST
SQLAlchemy      → persistenza in SQLite
Pydantic        → validazione e serializzazione
JWT + bcrypt    → autenticazione
CORS            → accesso dai frontend
Middleware      → logging, metriche
OpenAPI         → documentazione automatica
```

## Dominio del problema

```
UTENTI       → login / registrazione / ruoli (admin, instructor, alumno)
CORSI        → titolo, descrizione, capienza, stato, instructor
MODULI       → parti di un corso, con ordine
ISCRIZIONI   → alumno → corso, con stato (pendiente/activa/completada/cancelada)
VALUTAZIONI  → voto per modulo per alumno
```

In [ ]:
import subprocess, sys

for pkg in ['fastapi', 'uvicorn[standard]', 'sqlalchemy', 'python-jose[cryptography]', 'passlib[bcrypt]', 'python-multipart', 'requests']:
    try:
        __import__(pkg.split('[')[0].replace('-','_'))
        print(f"{pkg.split('[')[0]}: già installato")
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", pkg], check=True)
        print(f"{pkg}: installato")

---
## Livello 1: Database

In [ ]:
from sqlalchemy import (
    create_engine, Column, Integer, String, Float, Boolean,
    ForeignKey, DateTime, Text, Enum as SQLEnum
)
from sqlalchemy.orm import declarative_base, relationship, Session, sessionmaker
from datetime import datetime
import enum

DATABASE_URL = "sqlite:///./formaciones.db"
engine       = create_engine(DATABASE_URL, connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base         = declarative_base()


class RolUsuario(str, enum.Enum):
    admin      = "admin"
    instructor = "instructor"
    alumno     = "alumno"

class EstadoCurso(str, enum.Enum):
    borrador   = "borrador"
    publicado  = "publicado"
    cerrado    = "cerrado"

class EstadoInscripcion(str, enum.Enum):
    pendiente  = "pendiente"
    activa     = "activa"
    completada = "completada"
    cancelada  = "cancelada"


class Usuario(Base):
    __tablename__ = "usuarios"
    id            = Column(Integer, primary_key=True)
    username      = Column(String(50),  nullable=False, unique=True)
    nombre        = Column(String(100), nullable=False)
    email         = Column(String(200), nullable=False, unique=True)
    password_hash = Column(String(200), nullable=False)
    rol           = Column(SQLEnum(RolUsuario), default=RolUsuario.alumno)
    activo        = Column(Boolean, default=True)
    creado_en     = Column(DateTime, default=datetime.utcnow)

    cursos_instruidos = relationship("Curso",         back_populates="instructor")
    inscripciones     = relationship("Inscripcion",   back_populates="alumno")


class Curso(Base):
    __tablename__   = "cursos"
    id              = Column(Integer, primary_key=True)
    titulo          = Column(String(200), nullable=False)
    descripcion     = Column(Text, nullable=True)
    aforo           = Column(Integer, nullable=False, default=20)
    duracion_horas  = Column(Integer, nullable=True)
    precio          = Column(Float, nullable=True)
    estado          = Column(SQLEnum(EstadoCurso), default=EstadoCurso.borrador)
    creado_en       = Column(DateTime, default=datetime.utcnow)
    instructor_id   = Column(Integer, ForeignKey("usuarios.id"), nullable=True)

    instructor    = relationship("Usuario",      back_populates="cursos_instruidos")
    modulos       = relationship("Modulo",       back_populates="curso", cascade="all, delete-orphan", order_by="Modulo.orden")
    inscripciones = relationship("Inscripcion",  back_populates="curso",  cascade="all, delete-orphan")


class Modulo(Base):
    __tablename__  = "modulos"
    id             = Column(Integer, primary_key=True)
    titulo         = Column(String(200), nullable=False)
    descripcion    = Column(Text, nullable=True)
    orden          = Column(Integer, nullable=False, default=1)
    duracion_horas = Column(Integer, nullable=True)
    curso_id       = Column(Integer, ForeignKey("cursos.id", ondelete="CASCADE"))

    curso         = relationship("Curso",        back_populates="modulos")
    calificaciones= relationship("Calificacion", back_populates="modulo",  cascade="all, delete-orphan")


class Inscripcion(Base):
    __tablename__   = "inscripciones"
    id              = Column(Integer, primary_key=True)
    alumno_id       = Column(Integer, ForeignKey("usuarios.id"))
    curso_id        = Column(Integer, ForeignKey("cursos.id",  ondelete="CASCADE"))
    estado          = Column(SQLEnum(EstadoInscripcion), default=EstadoInscripcion.pendiente)
    inscrito_en     = Column(DateTime, default=datetime.utcnow)
    completado_en   = Column(DateTime, nullable=True)

    alumno = relationship("Usuario", back_populates="inscripciones")
    curso  = relationship("Curso",   back_populates="inscripciones")


class Calificacion(Base):
    __tablename__ = "calificaciones"
    id            = Column(Integer, primary_key=True)
    alumno_id     = Column(Integer, ForeignKey("usuarios.id"))
    modulo_id     = Column(Integer, ForeignKey("modulos.id",  ondelete="CASCADE"))
    nota          = Column(Float, nullable=False)
    comentario    = Column(Text, nullable=True)
    calificado_en = Column(DateTime, default=datetime.utcnow)

    alumno = relationship("Usuario")
    modulo = relationship("Modulo", back_populates="calificaciones")


Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)
print("Tabelle create:", engine.dialect.get_table_names(engine.connect()))

---
## Livello 2: Schemi Pydantic

In [ ]:
from pydantic import BaseModel, Field, EmailStr, ConfigDict
from typing import Optional

# ── Auth ─────────────────────────────────────────────────────
class RegistroRequest(BaseModel):
    username: str  = Field(min_length=3, max_length=50, pattern=r"^[a-z0-9_]+$")
    nombre:   str  = Field(min_length=1)
    email:    str  = Field(min_length=5)
    password: str  = Field(min_length=8)

class UsuarioResponse(BaseModel):
    id:       int
    username: str
    nombre:   str
    email:    str
    rol:      RolUsuario
    activo:   bool
    model_config = ConfigDict(from_attributes=True)

class Token(BaseModel):
    access_token: str
    token_type:   str = "bearer"

# ── Cursos ───────────────────────────────────────────────────
class CursoCreate(BaseModel):
    titulo:         str            = Field(min_length=1, max_length=200)
    descripcion:    Optional[str]  = None
    aforo:          int            = Field(default=20, ge=1)
    duracion_horas: Optional[int]  = Field(None, gt=0)
    precio:         Optional[float]= Field(None, ge=0)

class CursoUpdate(BaseModel):
    titulo:         Optional[str]   = Field(None, min_length=1)
    descripcion:    Optional[str]   = None
    aforo:          Optional[int]   = Field(None, ge=1)
    duracion_horas: Optional[int]   = Field(None, gt=0)
    precio:         Optional[float] = Field(None, ge=0)

class CursoResponse(BaseModel):
    id:              int
    titulo:          str
    descripcion:     Optional[str]
    aforo:           int
    duracion_horas:  Optional[int]
    precio:          Optional[float]
    estado:          EstadoCurso
    instructor_id:   Optional[int]
    inscritos:       int = 0
    plazas_libres:   int = 0
    model_config = ConfigDict(from_attributes=True)

class CursoDetalle(CursoResponse):
    modulos: list["ModuloResponse"] = []

# ── Moduli ──────────────────────────────────────────────────
class ModuloCreate(BaseModel):
    titulo:         str           = Field(min_length=1)
    descripcion:    Optional[str] = None
    orden:          int           = Field(default=1, ge=1)
    duracion_horas: Optional[int] = Field(None, gt=0)

class ModuloResponse(ModuloCreate):
    id:       int
    curso_id: int
    model_config = ConfigDict(from_attributes=True)

# ── Iscrizioni ─────────────────────────────────────────────
class InscripcionResponse(BaseModel):
    id:           int
    alumno_id:    int
    curso_id:     int
    estado:       EstadoInscripcion
    inscrito_en:  datetime
    model_config = ConfigDict(from_attributes=True)

# ── Valutazioni ────────────────────────────────────────────
class CalificacionCreate(BaseModel):
    alumno_id:  int
    nota:       float = Field(ge=0, le=10)
    comentario: Optional[str] = None

class CalificacionResponse(CalificacionCreate):
    id:            int
    modulo_id:     int
    calificado_en: datetime
    model_config = ConfigDict(from_attributes=True)

CursoDetalle.model_rebuild()
print("Schemi pronti")

---
## Livello 3: Autenticazione

In [ ]:
from passlib.context import CryptContext
from jose import jwt, JWTError
from datetime import datetime, timedelta, timezone
from fastapi import Depends, HTTPException, status
from fastapi.security import OAuth2PasswordBearer

pwd_context   = CryptContext(schemes=["bcrypt"], deprecated="auto")
SECRET_KEY    = "formaciones-secret-key-min-32-chars-seguro"
ALGORITHM     = "HS256"
EXP_MINUTOS   = 60

oauth2_scheme = OAuth2PasswordBearer(tokenUrl="/auth/token")


def hash_password(p: str)  -> str:  return pwd_context.hash(p)
def check_password(p, h)   -> bool: return pwd_context.verify(p, h)

def crear_token(data: dict) -> str:
    payload = {**data, "exp": datetime.now(timezone.utc) + timedelta(minutes=EXP_MINUTOS)}
    return jwt.encode(payload, SECRET_KEY, algorithm=ALGORITHM)


def get_db():
    db = SessionLocal()
    try:   yield db
    finally: db.close()


def usuario_actual(token: str = Depends(oauth2_scheme), db: Session = Depends(get_db)) -> Usuario:
    cred_exc = HTTPException(
        status_code=status.HTTP_401_UNAUTHORIZED,
        detail="Token non valido o scaduto",
        headers={"WWW-Authenticate": "Bearer"}
    )
    try:
        payload  = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        user_id  = payload.get("sub")
        if not user_id: raise cred_exc
    except JWTError:
        raise cred_exc

    usuario = db.get(Usuario, int(user_id))
    if not usuario or not usuario.activo:
        raise cred_exc
    return usuario


def require_admin(u: Usuario = Depends(usuario_actual)):
    if u.rol != RolUsuario.admin:
        raise HTTPException(403, "Solo amministratori")
    return u

def require_instructor_o_admin(u: Usuario = Depends(usuario_actual)):
    if u.rol not in (RolUsuario.instructor, RolUsuario.admin):
        raise HTTPException(403, "Solo instructor o amministratori")
    return u


print("Livello di autenticazione pronto")

---
## Livello 4: API FastAPI

In [ ]:
from fastapi import FastAPI, APIRouter, Depends, HTTPException, Query, Request, BackgroundTasks
from fastapi.middleware.cors import CORSMiddleware
from fastapi.security import OAuth2PasswordRequestForm
from fastapi.exceptions import RequestValidationError
from fastapi.responses import JSONResponse
from starlette.middleware.base import BaseHTTPMiddleware
from sqlalchemy import func
import time, uuid, logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")
log = logging.getLogger("formaciones")


app = FastAPI(
    title="API Formaciones",
    description="""
## API di gestione corsi e formazioni

Gestisce corsi, moduli, iscrizioni e valutazioni.

### Ruoli
- **admin**: accesso totale
- **instructor**: crea e gestisce i propri corsi
- **alumno**: si iscrive e consulta i propri dati

### Autenticazione
Bearer JWT. Ottieni il token su `/auth/token`.
""",
    version="1.0.0",
    openapi_tags=[
        {"name": "Auth",           "description": "Registrazione e login"},
        {"name": "Cursos",         "description": "Gestione dei corsi"},
        {"name": "Módulos",        "description": "Moduli di ogni corso"},
        {"name": "Inscripciones",  "description": "Iscrizioni degli alunni"},
        {"name": "Calificaciones", "description": "Voti per modulo"},
        {"name": "Estadísticas",   "description": "Riepiloghi e metriche"},
    ]
)

# CORS
app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:3000", "http://localhost:8080"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"]
)

# Middleware di logging
class LoggingMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request: Request, call_next):
        rid = str(uuid.uuid4())[:8]
        t0  = time.time()
        response = await call_next(request)
        ms  = (time.time() - t0) * 1000
        log.info(f"[{rid}] {request.method} {request.url.path} → {response.status_code} ({ms:.0f}ms)")
        response.headers["X-Request-ID"]    = rid
        response.headers["X-Response-Time"] = f"{ms:.0f}ms"
        return response

app.add_middleware(LoggingMiddleware)

# Gestori di errore
@app.exception_handler(RequestValidationError)
async def validation_handler(request: Request, exc: RequestValidationError):
    return JSONResponse(422, {"error": "VALIDATION_ERROR",
                               "message": "Dati non validi",
                               "details": exc.errors()})

@app.exception_handler(HTTPException)
async def http_handler(request: Request, exc: HTTPException):
    return JSONResponse(exc.status_code, {"error": f"HTTP_{exc.status_code}",
                                           "message": exc.detail, "details": None})


# ════════════════════════════════════════════════════════════
#  AUTH
# ════════════════════════════════════════════════════════════
auth_router = APIRouter(prefix="/auth", tags=["Auth"])

@auth_router.post("/registro", response_model=UsuarioResponse, status_code=201,
                  summary="Registra un nuovo utente")
def registro(datos: RegistroRequest, db: Session = Depends(get_db)):
    if db.query(Usuario).filter(Usuario.username == datos.username).first():
        raise HTTPException(409, "Username già esistente")
    if db.query(Usuario).filter(Usuario.email == datos.email).first():
        raise HTTPException(409, "Email già registrata")
    u = Usuario(username=datos.username, nombre=datos.nombre,
                email=datos.email, password_hash=hash_password(datos.password))
    db.add(u); db.commit(); db.refresh(u)
    return u

@auth_router.post("/token", response_model=Token, summary="Ottieni il token JWT")
def login(form: OAuth2PasswordRequestForm = Depends(), db: Session = Depends(get_db)):
    u = db.query(Usuario).filter(Usuario.username == form.username).first()
    if not u or not check_password(form.password, u.password_hash):
        raise HTTPException(401, "Credenziali errate")
    if not u.activo:
        raise HTTPException(403, "Account disattivato")
    return Token(access_token=crear_token({"sub": str(u.id), "rol": u.rol}))

@auth_router.get("/me", response_model=UsuarioResponse, summary="Il mio profilo")
def mi_perfil(u: Usuario = Depends(usuario_actual)):
    return u


# ════════════════════════════════════════════════════════════
#  CURSOS
# ════════════════════════════════════════════════════════════
cursos_router = APIRouter(prefix="/cursos", tags=["Cursos"])

def enriquecer_curso(curso: Curso, db: Session) -> dict:
    inscritos = db.query(func.count(Inscripcion.id)).filter(
        Inscripcion.curso_id == curso.id,
        Inscripcion.estado.in_([EstadoInscripcion.pendiente, EstadoInscripcion.activa])
    ).scalar()
    d = {c.name: getattr(curso, c.name) for c in curso.__table__.columns}
    d["inscritos"]     = inscritos
    d["plazas_libres"] = max(0, curso.aforo - inscritos)
    return d

@cursos_router.get("", response_model=list[CursoResponse], summary="Elenca i corsi")
def listar_cursos(
    estado:     Optional[EstadoCurso] = None,
    skip:       int = Query(0,  ge=0),
    limit:      int = Query(20, le=100),
    db:         Session = Depends(get_db)
):
    q = db.query(Curso)
    if estado: q = q.filter(Curso.estado == estado)
    cursos = q.offset(skip).limit(limit).all()
    return [enriquecer_curso(c, db) for c in cursos]

@cursos_router.get("/{curso_id}", response_model=CursoDetalle, summary="Ottieni un corso con i moduli")
def obtener_curso(curso_id: int, db: Session = Depends(get_db)):
    curso = db.get(Curso, curso_id)
    if not curso: raise HTTPException(404, "Corso non trovato")
    data = enriquecer_curso(curso, db)
    data["modulos"] = curso.modulos
    return data

@cursos_router.post("", response_model=CursoResponse, status_code=201,
                    summary="Crea un corso (instructor o admin)")
def crear_curso(
    datos: CursoCreate,
    u:     Usuario = Depends(require_instructor_o_admin),
    db:    Session = Depends(get_db)
):
    curso = Curso(**datos.model_dump(), instructor_id=u.id)
    db.add(curso); db.commit(); db.refresh(curso)
    return enriquecer_curso(curso, db)

@cursos_router.patch("/{curso_id}", response_model=CursoResponse, summary="Aggiorna un corso")
def actualizar_curso(
    curso_id: int,
    cambios:  CursoUpdate,
    u:        Usuario = Depends(require_instructor_o_admin),
    db:       Session = Depends(get_db)
):
    curso = db.get(Curso, curso_id)
    if not curso: raise HTTPException(404, "Corso non trovato")
    if u.rol != RolUsuario.admin and curso.instructor_id != u.id:
        raise HTTPException(403, "Solo l'instructor proprietario o admin")
    for k, v in cambios.model_dump(exclude_none=True).items():
        setattr(curso, k, v)
    db.commit(); db.refresh(curso)
    return enriquecer_curso(curso, db)

@cursos_router.post("/{curso_id}/publicar", summary="Pubblica il corso (admin o instructor proprietario)")
def publicar_curso(
    curso_id: int,
    u:        Usuario = Depends(require_instructor_o_admin),
    db:       Session = Depends(get_db)
):
    curso = db.get(Curso, curso_id)
    if not curso: raise HTTPException(404, "Corso non trovato")
    if u.rol != RolUsuario.admin and curso.instructor_id != u.id:
        raise HTTPException(403, "Non hai i permessi su questo corso")
    if not curso.modulos:
        raise HTTPException(400, "Un corso deve avere almeno un modulo per essere pubblicato")
    curso.estado = EstadoCurso.publicado
    db.commit()
    return {"mensaje": f"Corso '{curso.titulo}' pubblicato"}

@cursos_router.delete("/{curso_id}", status_code=204, summary="Elimina un corso (admin)")
def eliminar_curso(
    curso_id: int,
    _:        Usuario = Depends(require_admin),
    db:       Session = Depends(get_db)
):
    curso = db.get(Curso, curso_id)
    if not curso: raise HTTPException(404, "Corso non trovato")
    db.delete(curso); db.commit()


# ════════════════════════════════════════════════════════════
#  MÓDULOS
# ════════════════════════════════════════════════════════════
modulos_router = APIRouter(prefix="/cursos/{curso_id}/modulos", tags=["Módulos"])

@modulos_router.post("", response_model=ModuloResponse, status_code=201, summary="Aggiungi un modulo")
def crear_modulo(
    curso_id: int,
    datos:    ModuloCreate,
    u:        Usuario = Depends(require_instructor_o_admin),
    db:       Session = Depends(get_db)
):
    curso = db.get(Curso, curso_id)
    if not curso: raise HTTPException(404, "Corso non trovato")
    if u.rol != RolUsuario.admin and curso.instructor_id != u.id:
        raise HTTPException(403, "Solo l'instructor proprietario o admin")
    m = Modulo(**datos.model_dump(), curso_id=curso_id)
    db.add(m); db.commit(); db.refresh(m)
    return m

@modulos_router.delete("/{modulo_id}", status_code=204, summary="Elimina un modulo")
def eliminar_modulo(
    curso_id:  int,
    modulo_id: int,
    u:         Usuario = Depends(require_instructor_o_admin),
    db:        Session = Depends(get_db)
):
    m = db.get(Modulo, modulo_id)
    if not m or m.curso_id != curso_id: raise HTTPException(404, "Modulo non trovato")
    db.delete(m); db.commit()


# ════════════════════════════════════════════════════════════
#  INSCRIPCIONES
# ════════════════════════════════════════════════════════════
insc_router = APIRouter(prefix="/cursos/{curso_id}/inscripciones", tags=["Inscripciones"])

@insc_router.post("", response_model=InscripcionResponse, status_code=201,
                  summary="Iscriviti a un corso")
def inscribirse(
    curso_id: int,
    u:        Usuario = Depends(usuario_actual),
    db:       Session = Depends(get_db)
):
    curso = db.get(Curso, curso_id)
    if not curso: raise HTTPException(404, "Corso non trovato")
    if curso.estado != EstadoCurso.publicado:
        raise HTTPException(400, "Puoi iscriverti solo a corsi pubblicati")

    # Verificare se è già iscritto
    ya = db.query(Inscripcion).filter(
        Inscripcion.alumno_id == u.id, Inscripcion.curso_id == curso_id,
        Inscripcion.estado.in_([EstadoInscripcion.pendiente, EstadoInscripcion.activa])
    ).first()
    if ya: raise HTTPException(409, "Sei già iscritto a questo corso")

    # Verificare la capienza
    inscritos = db.query(func.count(Inscripcion.id)).filter(
        Inscripcion.curso_id == curso_id,
        Inscripcion.estado.in_([EstadoInscripcion.pendiente, EstadoInscripcion.activa])
    ).scalar()
    if inscritos >= curso.aforo:
        raise HTTPException(409, "Corso al completo, nessun posto disponibile")

    insc = Inscripcion(alumno_id=u.id, curso_id=curso_id)
    db.add(insc); db.commit(); db.refresh(insc)
    return insc

@insc_router.get("", response_model=list[InscripcionResponse],
                 summary="Visualizza le iscrizioni (instructor o admin)")
def listar_inscripciones(
    curso_id: int,
    u:        Usuario = Depends(require_instructor_o_admin),
    db:       Session = Depends(get_db)
):
    return db.query(Inscripcion).filter(Inscripcion.curso_id == curso_id).all()

@insc_router.patch("/{insc_id}/estado", response_model=InscripcionResponse,
                   summary="Cambia lo stato di un'iscrizione")
def cambiar_estado(
    curso_id: int,
    insc_id:  int,
    estado:   EstadoInscripcion,
    u:        Usuario = Depends(require_instructor_o_admin),
    db:       Session = Depends(get_db)
):
    insc = db.get(Inscripcion, insc_id)
    if not insc or insc.curso_id != curso_id: raise HTTPException(404, "Iscrizione non trovata")
    insc.estado = estado
    if estado == EstadoInscripcion.completada:
        insc.completado_en = datetime.utcnow()
    db.commit(); db.refresh(insc)
    return insc


# ════════════════════════════════════════════════════════════
#  CALIFICACIONES
# ════════════════════════════════════════════════════════════
cal_router = APIRouter(prefix="/modulos/{modulo_id}/calificaciones", tags=["Calificaciones"])

@cal_router.post("", response_model=CalificacionResponse, status_code=201,
                 summary="Aggiungi o aggiorna una valutazione")
def calificar(
    modulo_id: int,
    datos:     CalificacionCreate,
    u:         Usuario = Depends(require_instructor_o_admin),
    db:        Session = Depends(get_db)
):
    m = db.get(Modulo, modulo_id)
    if not m: raise HTTPException(404, "Modulo non trovato")

    # Verificare che l'alunno sia iscritto al corso
    insc = db.query(Inscripcion).filter(
        Inscripcion.alumno_id == datos.alumno_id,
        Inscripcion.curso_id  == m.curso_id,
        Inscripcion.estado    == EstadoInscripcion.activa
    ).first()
    if not insc: raise HTTPException(400, "L'alunno non ha un'iscrizione attiva a questo corso")

    # Upsert della valutazione
    cal = db.query(Calificacion).filter(
        Calificacion.alumno_id == datos.alumno_id,
        Calificacion.modulo_id == modulo_id
    ).first()
    if cal:
        cal.nota       = datos.nota
        cal.comentario = datos.comentario
    else:
        cal = Calificacion(modulo_id=modulo_id, **datos.model_dump())
        db.add(cal)
    db.commit(); db.refresh(cal)
    return cal

@cal_router.get("", response_model=list[CalificacionResponse], summary="Visualizza le valutazioni")
def ver_calificaciones(
    modulo_id: int,
    u:         Usuario = Depends(require_instructor_o_admin),
    db:        Session = Depends(get_db)
):
    return db.query(Calificacion).filter(Calificacion.modulo_id == modulo_id).all()


# ════════════════════════════════════════════════════════════
#  ESTADÍSTICAS
# ════════════════════════════════════════════════════════════
stats_router = APIRouter(prefix="/estadisticas", tags=["Estadísticas"])

@stats_router.get("/resumen", summary="Riepilogo generale")
def resumen(u: Usuario = Depends(require_admin), db: Session = Depends(get_db)):
    return {
        "usuarios":          db.query(func.count(Usuario.id)).scalar(),
        "cursos":            db.query(func.count(Curso.id)).scalar(),
        "cursos_publicados": db.query(func.count(Curso.id)).filter(Curso.estado == EstadoCurso.publicado).scalar(),
        "inscripciones":     db.query(func.count(Inscripcion.id)).scalar(),
        "inscripciones_activas": db.query(func.count(Inscripcion.id)).filter(Inscripcion.estado == EstadoInscripcion.activa).scalar(),
        "nota_media":        round(db.query(func.avg(Calificacion.nota)).scalar() or 0, 2),
    }

@stats_router.get("/cursos/{curso_id}", summary="Statistiche di un corso")
def stats_curso(
    curso_id: int,
    u:        Usuario = Depends(require_instructor_o_admin),
    db:       Session = Depends(get_db)
):
    curso = db.get(Curso, curso_id)
    if not curso: raise HTTPException(404, "Corso non trovato")

    inscritos   = db.query(func.count(Inscripcion.id)).filter(Inscripcion.curso_id == curso_id).scalar()
    completados = db.query(func.count(Inscripcion.id)).filter(
        Inscripcion.curso_id == curso_id, Inscripcion.estado == EstadoInscripcion.completada
    ).scalar()
    nota_media  = db.query(func.avg(Calificacion.nota)).join(Modulo).filter(
        Modulo.curso_id == curso_id
    ).scalar()

    return {
        "curso":         curso.titulo,
        "aforo":         curso.aforo,
        "inscritos":     inscritos,
        "completados":   completados,
        "tasa_exito":    f"{100 * completados / inscritos:.1f}%" if inscritos else "0%",
        "nota_media":    round(nota_media or 0, 2),
        "modulos":       len(curso.modulos),
    }


# Registrare i router
for router in [auth_router, cursos_router, modulos_router, insc_router, cal_router, stats_router]:
    app.include_router(router)

print("API costruita. Endpoint:")
for r in app.routes:
    if hasattr(r, 'methods'):
        print(f"  {list(r.methods)[0]:6} {r.path}")

---
## Demo completo

In [ ]:
import uvicorn, threading, time, requests as req, json

cfg = uvicorn.Config(app, host="127.0.0.1", port=8040, log_level="warning")
threading.Thread(target=uvicorn.Server(cfg).run, daemon=True).start()
time.sleep(1.5)
B = "http://127.0.0.1:8040"
print(f"API su {B}")
print(f"Docs: {B}/docs")

In [ ]:
# ── Registrare gli utenti ───────────────────────────────────────
print("=== 1. Registrare gli utenti ===")

admin_data = req.post(f"{B}/auth/registro", json={
    "username": "admin1", "nombre": "Admin Sistema",
    "email": "admin@formaciones.com", "password": "adminPass123"
}).json()
print(f"  Registrato: {admin_data['nombre']} (id={admin_data['id']})")

inst_data = req.post(f"{B}/auth/registro", json={
    "username": "prof_ana", "nombre": "Ana Instructora",
    "email": "ana@formaciones.com", "password": "anaPass123"
}).json()

alu1_data = req.post(f"{B}/auth/registro", json={
    "username": "carlos_a", "nombre": "Carlos Alumno",
    "email": "carlos@empresa.com", "password": "carlosPass123"
}).json()

alu2_data = req.post(f"{B}/auth/registro", json={
    "username": "lucia_b", "nombre": "Lucía Alumna",
    "email": "lucia@empresa.com", "password": "luciaPass123"
}).json()

print(f"  Utenti creati: admin1, prof_ana, carlos_a, lucia_b")

# Admin fa il login
token_admin = req.post(f"{B}/auth/token",
    data={"username": "admin1", "password": "adminPass123"}).json()["access_token"]
H_ADMIN = {"Authorization": f"Bearer {token_admin}"}

# Aggiornare i ruoli direttamente nel DB (solo per la demo)
with SessionLocal() as db:
    u_admin = db.get(Usuario, admin_data['id'])
    u_admin.rol = RolUsuario.admin
    u_inst  = db.get(Usuario, inst_data['id'])
    u_inst.rol  = RolUsuario.instructor
    db.commit()
print("  Ruoli assegnati: admin1=admin, prof_ana=instructor")

# Ri-effettuare il login per ottenere i token con il ruolo aggiornato
token_admin = req.post(f"{B}/auth/token",
    data={"username": "admin1", "password": "adminPass123"}).json()["access_token"]
token_inst  = req.post(f"{B}/auth/token",
    data={"username": "prof_ana", "password": "anaPass123"}).json()["access_token"]
token_alu1  = req.post(f"{B}/auth/token",
    data={"username": "carlos_a", "password": "carlosPass123"}).json()["access_token"]
token_alu2  = req.post(f"{B}/auth/token",
    data={"username": "lucia_b", "password": "luciaPass123"}).json()["access_token"]

H_ADMIN = {"Authorization": f"Bearer {token_admin}"}
H_INST  = {"Authorization": f"Bearer {token_inst}"}
H_ALU1  = {"Authorization": f"Bearer {token_alu1}"}
H_ALU2  = {"Authorization": f"Bearer {token_alu2}"}

In [ ]:
# ── Creare corsi e moduli ────────────────────────────────────
print("=== 2. Creare i corsi ===")

curso1 = req.post(f"{B}/cursos", headers=H_INST, json={
    "titulo":        "Python per l'analisi dei dati",
    "descripcion":   "Corso intensivo di Python, pandas e visualizzazione",
    "aforo":         15,
    "duracion_horas":40,
    "precio":        299.0
}).json()
print(f"  Corso 1: {curso1['titulo']} (id={curso1['id']}, estado={curso1['estado']})")

curso2 = req.post(f"{B}/cursos", headers=H_ADMIN, json={
    "titulo":        "FastAPI da zero",
    "descripcion":   "Costruzione di API REST con FastAPI",
    "aforo":         3,  # capienza piccola per testare il limite
    "duracion_horas":20,
    "precio":        199.0
}).json()
print(f"  Corso 2: {curso2['titulo']} (id={curso2['id']}, aforo={curso2['aforo']})")

print("\n=== 3. Aggiungere i moduli ===")
for i, titulo in enumerate(["Introduzione e ambiente", "pandas base", "Visualizzazione"], 1):
    req.post(f"{B}/cursos/{curso1['id']}/modulos", headers=H_INST,
             json={"titulo": titulo, "orden": i, "duracion_horas": 13})

m1 = req.post(f"{B}/cursos/{curso2['id']}/modulos", headers=H_ADMIN,
              json={"titulo": "HTTP e REST", "orden": 1, "duracion_horas": 5}).json()
m2 = req.post(f"{B}/cursos/{curso2['id']}/modulos", headers=H_ADMIN,
              json={"titulo": "FastAPI base", "orden": 2, "duracion_horas": 8}).json()

detalle = req.get(f"{B}/cursos/{curso1['id']}").json()
print(f"  Il Corso 1 ha {len(detalle['modulos'])} moduli")

print("\n=== 4. Pubblicare i corsi ===")
r1 = req.post(f"{B}/cursos/{curso1['id']}/publicar", headers=H_INST)
r2 = req.post(f"{B}/cursos/{curso2['id']}/publicar", headers=H_ADMIN)
print(f"  Corso 1: {r1.json()['mensaje']}")
print(f"  Corso 2: {r2.json()['mensaje']}")

In [ ]:
# ── Iscrizioni e valutazioni ───────────────────────────
print("=== 5. Iscrizioni ===")

# Carlos e Lucía si iscrivono al curso1
i1 = req.post(f"{B}/cursos/{curso1['id']}/inscripciones", headers=H_ALU1).json()
print(f"  Carlos → Corso 1: {i1['estado']}")

i2 = req.post(f"{B}/cursos/{curso1['id']}/inscripciones", headers=H_ALU2).json()
print(f"  Lucía  → Corso 1: {i2['estado']}")

# Carlos prova a iscriversi due volte
r = req.post(f"{B}/cursos/{curso1['id']}/inscripciones", headers=H_ALU1)
print(f"  Carlos duplicato: {r.status_code} — {r.json()['message']}")

# Riempire la capienza del curso2 (aforo=3)
req.post(f"{B}/cursos/{curso2['id']}/inscripciones", headers=H_ALU1)
req.post(f"{B}/cursos/{curso2['id']}/inscripciones", headers=H_ALU2)
# Registrare un terzo alunno
req.post(f"{B}/auth/registro", json={"username": "pedro_c", "nombre": "Pedro",
    "email": "pedro@x.com", "password": "pedroPass123"})
token_alu3 = req.post(f"{B}/auth/token", data={"username": "pedro_c", "password": "pedroPass123"}).json()["access_token"]
H_ALU3 = {"Authorization": f"Bearer {token_alu3}"}
req.post(f"{B}/cursos/{curso2['id']}/inscripciones", headers=H_ALU3)

# Ora il curso2 è al completo
req.post(f"{B}/auth/registro", json={"username": "extra", "nombre": "Extra",
    "email": "extra@x.com", "password": "extraPass123"})
token_extra = req.post(f"{B}/auth/token", data={"username": "extra", "password": "extraPass123"}).json()["access_token"]
r = req.post(f"{B}/cursos/{curso2['id']}/inscripciones", headers={"Authorization": f"Bearer {token_extra}"})
print(f"  Capienza esaurita: {r.status_code} — {r.json()['message']}")

print("\n=== 6. Approvare le iscrizioni e valutare ===")
# Approvare le iscrizioni di carlos e lucia al curso1
req.patch(f"{B}/cursos/{curso1['id']}/inscripciones/{i1['id']}/estado",
          headers=H_INST, params={"estado": "activa"})
req.patch(f"{B}/cursos/{curso1['id']}/inscripciones/{i2['id']}/estado",
          headers=H_INST, params={"estado": "activa"})

# Valutare nel primo modulo
modulo_id = detalle['modulos'][0]['id']
cal1 = req.post(f"{B}/modulos/{modulo_id}/calificaciones", headers=H_INST, json={
    "alumno_id": alu1_data['id'], "nota": 8.5, "comentario": "Buon lavoro"
}).json()
cal2 = req.post(f"{B}/modulos/{modulo_id}/calificaciones", headers=H_INST, json={
    "alumno_id": alu2_data['id'], "nota": 9.2
}).json()
print(f"  Carlos: {cal1['nota']} | Lucía: {cal2['nota']}")

In [ ]:
# ── Statistiche ─────────────────────────────────────────────
print("=== 7. Statistiche ===")

resumen = req.get(f"{B}/estadisticas/resumen", headers=H_ADMIN).json()
print("Riepilogo generale:")
for k, v in resumen.items():
    print(f"  {k}: {v}")

print(f"\nStatistiche Corso 1 ({curso1['titulo']}):")
stats = req.get(f"{B}/estadisticas/cursos/{curso1['id']}", headers=H_INST).json()
for k, v in stats.items():
    print(f"  {k}: {v}")

print(f"\nElenco dei corsi pubblicati:")
cursos_pub = req.get(f"{B}/cursos", params={"estado": "publicado"}).json()
for c in cursos_pub:
    print(f"  {c['titulo']}: {c['inscritos']}/{c['aforo']} iscritti, {c['plazas_libres']} liberi")

print(f"\n✓ API completa funzionante correttamente")
print(f"  Docs interattive: http://127.0.0.1:8040/docs")

---
## Esercizi di ampliamento

### Esercizio 1 — Certificati

**Consegna:** Aggiungi un sistema di certificati di completamento:

- Un alunno ottiene un certificato quando tutte le sue iscrizioni attive a un corso hanno stato `completada`
- `POST /cursos/{id}/certificados/{alumno_id}` — genera il certificato (solo instructor o admin)
- Modello `Certificado`: id, alumno_id, curso_id, nota_media, emitido_en, codigo_verificacion (UUID)
- `GET /certificados/{codigo}` — verifica pubblica del certificato

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
import uuid

class Certificado(Base):
    __tablename__       = "certificados"
    id                  = Column(Integer, primary_key=True)
    alumno_id           = Column(Integer, ForeignKey("usuarios.id"))
    curso_id            = Column(Integer, ForeignKey("cursos.id"))
    nota_media          = Column(Float)
    emitido_en          = Column(DateTime, default=datetime.utcnow)
    codigo_verificacion = Column(String(36), unique=True, default=lambda: str(uuid.uuid4()))

# Aggiungere all'app
@app.post("/cursos/{curso_id}/certificados/{alumno_id}")
def emitir_certificado(curso_id: int, alumno_id: int,
                        u: Usuario = Depends(require_instructor_o_admin),
                        db: Session = Depends(get_db)):
    # Verificare l'iscrizione completata
    insc = db.query(Inscripcion).filter(
        Inscripcion.alumno_id == alumno_id,
        Inscripcion.curso_id  == curso_id,
        Inscripcion.estado    == EstadoInscripcion.completada
    ).first()
    if not insc: raise HTTPException(400, "L'alunno non ha completato il corso")

    # Calcolare la nota media
    curso = db.get(Curso, curso_id)
    nota_media = db.query(func.avg(Calificacion.nota)).join(Modulo).filter(
        Modulo.curso_id   == curso_id,
        Calificacion.alumno_id == alumno_id
    ).scalar() or 0

    cert = Certificado(alumno_id=alumno_id, curso_id=curso_id,
                       nota_media=round(nota_media, 2))
    db.add(cert); db.commit(); db.refresh(cert)
    return {"certificado": cert.codigo_verificacion, "nota_media": cert.nota_media,
            "curso": curso.titulo, "emitido_en": cert.emitido_en}

@app.get("/certificados/{codigo}")
def verificar_certificado(codigo: str, db: Session = Depends(get_db)):
    cert = db.query(Certificado).filter(Certificado.codigo_verificacion == codigo).first()
    if not cert: raise HTTPException(404, "Certificato non trovato")
    alumno = db.get(Usuario, cert.alumno_id)
    curso  = db.get(Curso,   cert.curso_id)
    return {"valido": True, "alumno": alumno.nombre, "curso": curso.titulo,
            "nota_media": cert.nota_media, "emitido_en": cert.emitido_en}
```
</details>

---
### Esercizio 2 — La mia area alunno

**Consegna:** Aggiungi endpoint specifici per la vista dell'alunno:

- `GET /mis-cursos` — corsi a cui l'alunno autenticato è iscritto (con stato e progresso)
- `GET /mis-calificaciones` — tutti i suoi voti raggruppati per corso e modulo
- `GET /mis-certificados` — certificati ottenuti

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
@app.get("/mis-cursos", tags=["Alumno"])
def mis_cursos(u: Usuario = Depends(usuario_actual), db: Session = Depends(get_db)):
    inscripciones = db.query(Inscripcion).filter(Inscripcion.alumno_id == u.id).all()
    resultado = []
    for insc in inscripciones:
        curso = db.get(Curso, insc.curso_id)
        n_modulos = len(curso.modulos)
        n_calificados = db.query(func.count(Calificacion.id)).join(Modulo).filter(
            Modulo.curso_id == insc.curso_id, Calificacion.alumno_id == u.id
        ).scalar()
        resultado.append({
            "curso":       curso.titulo,
            "estado":      insc.estado,
            "progreso":    f"{n_calificados}/{n_modulos} moduli valutati",
            "inscrito_en": insc.inscrito_en
        })
    return resultado

@app.get("/mis-calificaciones", tags=["Alumno"])
def mis_calificaciones(u: Usuario = Depends(usuario_actual), db: Session = Depends(get_db)):
    cals = db.query(Calificacion).filter(Calificacion.alumno_id == u.id).all()
    por_curso = {}
    for c in cals:
        modulo = db.get(Modulo, c.modulo_id)
        curso  = db.get(Curso,  modulo.curso_id)
        por_curso.setdefault(curso.titulo, [])
        por_curso[curso.titulo].append({"modulo": modulo.titulo, "nota": c.nota})
    return por_curso
```
</details>

---
### Esercizio 3 — Ricerca di corsi

**Consegna:** Aggiungi un endpoint di ricerca avanzata dei corsi:

- `GET /cursos/buscar`
- Filtri: `q` (testo nel titolo o nella descrizione), `precio_max`, `duracion_max`, `plazas_disponibles=true`
- Ordinamento: `precio_asc`, `precio_desc`, `duracion_asc`, `mas_inscritos`
- Restituisce solo corsi pubblicati

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
from sqlalchemy import asc, desc
from typing import Literal, Optional

@app.get("/cursos/buscar", tags=["Cursos"])
def buscar_cursos(
    q:                   Optional[str]   = None,
    precio_max:          Optional[float] = Query(None, gt=0),
    duracion_max:        Optional[int]   = Query(None, gt=0),
    plazas_disponibles:  bool            = False,
    ordenar:             Literal["precio_asc","precio_desc","duracion_asc","mas_inscritos"] = "precio_asc",
    skip:                int             = 0,
    limit:               int             = Query(20, le=100),
    db:                  Session         = Depends(get_db)
):
    consulta = db.query(Curso).filter(Curso.estado == EstadoCurso.publicado)

    if q:            consulta = consulta.filter(Curso.titulo.ilike(f"%{q}%") | Curso.descripcion.ilike(f"%{q}%"))
    if precio_max:   consulta = consulta.filter(Curso.precio <= precio_max)
    if duracion_max: consulta = consulta.filter(Curso.duracion_horas <= duracion_max)

    cursos = consulta.all()

    # Arricchire con i dati degli iscritti
    enriquecidos = []
    for c in cursos:
        datos = enriquecer_curso(c, db)
        if plazas_disponibles and datos["plazas_libres"] == 0:
            continue
        enriquecidos.append(datos)

    # Ordinare
    if ordenar == "precio_asc":    enriquecidos.sort(key=lambda x: x.get("precio") or 0)
    elif ordenar == "precio_desc": enriquecidos.sort(key=lambda x: x.get("precio") or 0, reverse=True)
    elif ordenar == "duracion_asc": enriquecidos.sort(key=lambda x: x.get("duracion_horas") or 0)
    elif ordenar == "mas_inscritos": enriquecidos.sort(key=lambda x: x["inscritos"], reverse=True)

    total = len(enriquecidos)
    return {"total": total, "items": enriquecidos[skip:skip+limit]}
```
</details>

---
### Esercizio 4 — Notifiche pendenti

**Consegna:** Implementa un sistema di notifiche interne:

- Quando un alunno si iscrive → notificare l'instructor
- Quando viene assegnata una valutazione → notificare l'alunno
- Modello `Notificacion`: id, usuario_id, mensaje, leida (bool), creada_en
- `GET /mis-notificaciones` — elenca quelle non lette dell'utente autenticato
- `POST /mis-notificaciones/{id}/leer` — segna come letta

Usa `BackgroundTasks` per creare le notifiche.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
class Notificacion(Base):
    __tablename__ = "notificaciones"
    id         = Column(Integer, primary_key=True)
    usuario_id = Column(Integer, ForeignKey("usuarios.id"))
    mensaje    = Column(Text, nullable=False)
    leida      = Column(Boolean, default=False)
    creada_en  = Column(DateTime, default=datetime.utcnow)

def crear_notificacion(usuario_id: int, mensaje: str):
    with SessionLocal() as db:
        notif = Notificacion(usuario_id=usuario_id, mensaje=mensaje)
        db.add(notif); db.commit()

# Modificare l'endpoint di iscrizione per aggiungere un BackgroundTask:
# bg_tasks.add_task(crear_notificacion, curso.instructor_id,
#     f"Nuova iscrizione di {u.nombre} a '{curso.titulo}'")

@app.get("/mis-notificaciones", tags=["Alumno"])
def mis_notificaciones(u: Usuario = Depends(usuario_actual), db: Session = Depends(get_db)):
    return db.query(Notificacion).filter(
        Notificacion.usuario_id == u.id, Notificacion.leida == False
    ).order_by(Notificacion.creada_en.desc()).all()

@app.post("/mis-notificaciones/{notif_id}/leer", tags=["Alumno"])
def leer_notificacion(notif_id: int, u: Usuario = Depends(usuario_actual),
                      db: Session = Depends(get_db)):
    notif = db.get(Notificacion, notif_id)
    if not notif or notif.usuario_id != u.id: raise HTTPException(404, "Notifica non trovata")
    notif.leida = True; db.commit()
    return {"mensaje": "Notifica segnata come letta"}
```
</details>

---
### Esercizio 5 (sfida) — Esportazione dei dati

**Consegna:** Aggiungi funzionalità di esportazione per gli amministratori:

- `GET /admin/exportar/cursos.csv` — restituisce un CSV con tutti i corsi e le loro statistiche
- `GET /admin/exportar/inscripciones.csv` — CSV con alunno, corso, stato, nota media
- Usa `StreamingResponse` di FastAPI con un generatore che produce le righe del CSV
- Aggiungi l'header `Content-Disposition: attachment; filename=cursos.csv`

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
from fastapi.responses import StreamingResponse
import csv, io

@app.get("/admin/exportar/cursos.csv", tags=["Estadísticas"])
def exportar_cursos(_: Usuario = Depends(require_admin), db: Session = Depends(get_db)):
    def generar():
        buf = io.StringIO()
        writer = csv.writer(buf)
        writer.writerow(["id","titulo","estado","aforo","inscritos","precio","duracion_horas"])
        yield buf.getvalue(); buf.seek(0); buf.truncate()

        for c in db.query(Curso).all():
            datos = enriquecer_curso(c, db)
            writer.writerow([c.id, c.titulo, c.estado, c.aforo,
                             datos["inscritos"], c.precio, c.duracion_horas])
            yield buf.getvalue(); buf.seek(0); buf.truncate()

    return StreamingResponse(
        generar(), media_type="text/csv",
        headers={"Content-Disposition": "attachment; filename=cursos.csv"}
    )

@app.get("/admin/exportar/inscripciones.csv", tags=["Estadísticas"])
def exportar_inscripciones(_: Usuario = Depends(require_admin), db: Session = Depends(get_db)):
    def generar():
        buf = io.StringIO()
        writer = csv.writer(buf)
        writer.writerow(["alumno","email","curso","estado","inscrito_en","nota_media"])
        yield buf.getvalue(); buf.seek(0); buf.truncate()

        for insc in db.query(Inscripcion).all():
            alumno = db.get(Usuario, insc.alumno_id)
            curso  = db.get(Curso,   insc.curso_id)
            nota_media = db.query(func.avg(Calificacion.nota)).join(Modulo).filter(
                Modulo.curso_id == insc.curso_id, Calificacion.alumno_id == insc.alumno_id
            ).scalar()
            writer.writerow([alumno.nombre, alumno.email, curso.titulo, insc.estado,
                             insc.inscrito_en.date(), round(nota_media or 0, 2)])
            yield buf.getvalue(); buf.seek(0); buf.truncate()

    return StreamingResponse(
        generar(), media_type="text/csv",
        headers={"Content-Disposition": "attachment; filename=inscripciones.csv"}
    )
```
</details>

---
## Riepilogo del modulo 5

```
M5_C1: HTTP — protocollo, request/response, status code, header
M5_C2: REST + requests — principi REST, consumo di API esterne
M5_C3: FastAPI base — endpoint, path/query params, Pydantic, CRUD in memoria
M5_C4: FastAPI avanzato — SQLAlchemy, Depends, CRUD persistente, relazioni
M5_C5: Autenticazione — API key, bcrypt, JWT, OAuth2 Password Flow, ruoli
M5_C6: Docs e best practice — OpenAPI, CORS, middleware, rate limiting, versionamento
M5_C7: Progetto — API completa di gestione formazioni
```

### Stack completo padroneggiato

```python
from fastapi import FastAPI, Depends, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.security import OAuth2PasswordBearer
from sqlalchemy.orm import Session
from pydantic import BaseModel
from passlib.context import CryptContext
from jose import jwt

# → API REST professionale con autenticazione e database
```

### Modulo 6: cosa segue

Nel **Modulo 6** facciamo il salto verso l'intelligenza artificiale: modelli linguistici, embedding, RAG e agenti.